<a href="https://colab.research.google.com/github/Abhi0129n/gpu-computing-/blob/main/3d%20matrix%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
%%writefile matrix_add_3d.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define X 4
#define Y 4
#define Z 4

// CUDA Kernel
__global__ void matrixAdd3D(float *A, float *B, float *C,
                            int xSize, int ySize, int zSize)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    if (x < xSize && y < ySize && z < zSize)
    {
        int index = z * (xSize * ySize) + y * xSize + x;

        C[index] = A[index] + B[index];
    }
}

int main()
{
    int N = X * Y * Z;
    int size = N * sizeof(float);

    // Host memory
    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    // Initialize matrices
    for (int z = 0; z < Z; z++)
    {
        for (int y = 0; y < Y; y++)
        {
            for (int x = 0; x < X; x++)
            {
                int index = z * (X * Y) + y * X + x;

                h_A[index] = index;
                h_B[index] = index * 2;
            }
        }
    }

    // Device memory
    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // Copy CPU → GPU
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // 3D block
    dim3 threadsPerBlock(2, 2, 2);

    // 3D grid
    dim3 blocksPerGrid(
        (X + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (Y + threadsPerBlock.y - 1) / threadsPerBlock.y,
        (Z + threadsPerBlock.z - 1) / threadsPerBlock.z
    );

    // Launch kernel
    matrixAdd3D<<<blocksPerGrid, threadsPerBlock>>>(
        d_A, d_B, d_C, X, Y, Z
    );

    // Wait for GPU
    cudaDeviceSynchronize();

    // Copy GPU → CPU
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // Display result layer by layer
    for (int z = 0; z < Z; z++)
    {
        printf("\nLayer Z = %d\n", z);

        for (int y = 0; y < Y; y++)
        {
            for (int x = 0; x < X; x++)
            {
                int index = z * (X * Y) + y * X + x;

                printf("%.0f ", h_C[index]);
            }

            printf("\n");
        }
    }

    // Free GPU memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    // Free CPU memory
    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Overwriting matrix_add_3d.cu


In [21]:
!nvcc matrix_add_3d.cu -o matrix_add_3d

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [22]:
!./matrix_add_3d


Layer Z = 0
0 3 6 9 
12 15 18 21 
24 27 30 33 
36 39 42 45 

Layer Z = 1
48 51 54 57 
60 63 66 69 
72 75 78 81 
84 87 90 93 

Layer Z = 2
96 99 102 105 
108 111 114 117 
120 123 126 129 
132 135 138 141 

Layer Z = 3
144 147 150 153 
156 159 162 165 
168 171 174 177 
180 183 186 189 
